# Titanic Survival Prediction


*   This notbook would be divided into parts and eacvh part is gonna be explained
*   It will also have a part in which i would write the questions that i had throughout solving it and answer them



# Setup Kaggle API and download dataset

In [ ]:
from google.colab import files  #uploading kaggle.json so that kaggle API is set and authenticated(I would be able to download kaggle datasets directly withoput google drive)
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"selmachelbabi","key":"182db91e93122321f0e7539d6fca0680"}'}

In [ ]:
import os # creates kaggle configuration folder, moves API key there and secures the file

os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c titanic
!unzip titanic.zip

titanic.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  titanic.zip
replace gender_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


# Reading & Cleaning Data

In [ ]:
import pandas as pd
df=pd.read_csv('train.csv')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [ ]:
df.info()#shows  information about the data

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


# Checking for nulls, filling them & dropping duplicates

In [ ]:
df.isnull().any()# shows the categories that contain null values

,0
PassengerId,False
Survived,False
Pclass,False
Name,False
Sex,False
Age,True
SibSp,False
Parch,False
Ticket,False
Fare,False


The age, cabin and embarked categories have null values. I think we can fill age with the median age, while cabin and embarked with the mode.


In [ ]:
#replacing the null values of the column Age
median=df["Age"].median()#calculates the medain for the age column
df["Age"]=df["Age"].fillna(value=median)#replaces the null values with the median
df.isnull().any()#shows the columns that have null to check if age is still one of them ,which it is not
df["Age"].isnull()

,Age
0,False
1,False
2,False
3,False
4,False
...,...
886,False
887,False
888,False
889,False


In [ ]:
#repalcing the null values of the cabin column

mode_c=df["Cabin"].mode()[0]
df["Cabin"]=df["Cabin"].fillna(value=mode_c)
df["Cabin"].isnull()

,Cabin
0,False
1,False
2,False
3,False
4,False
...,...
886,False
887,False
888,False
889,False


In [ ]:
mode_e=df["Embarked"].mode()[0]
df["Embarked"]=df["Embarked"].fillna(value=mode_e)
df["Embarked"].isnull()

,Embarked
0,False
1,False
2,False
3,False
4,False
...,...
886,False
887,False
888,False
889,False


In [ ]:
df.isnull().any()

,0
PassengerId,False
Survived,False
Pclass,False
Name,False
Sex,False
Age,False
SibSp,False
Parch,False
Ticket,False
Fare,False


In [ ]:
df.drop_duplicates()
print(df.duplicated().sum())

0


# Checking/Correcting data types

In [ ]:
n=df["Name"].map(type).unique()
s=df["Sex"].map(type).unique()
t=df["Ticket"].map(type).unique()
c=df["Cabin"].map(type).unique()
e=df["Embarked"].map(type).unique()

print(n), print(s), print(t),print(c),print(e)

[<class 'str'>]
[<class 'str'>]
[<class 'str'>]
[<class 'str'>]
[<class 'str'>]


(None, None, None, None, None)

# Feature engineering

In [ ]:
df["Title"]=df["Name"].str.split(",").str[1].str.split('.').str[0].str.strip()#basically extracts the titles from names and strip any extra spaces
df["Ticketcount"]=df.groupby("Ticket")["Ticket"].transform('count')# groups passengesrrs that had the same tickets /basically they were traveling together or not
df["Ticketprefix"]=df["Ticket"].apply(lambda x: x.split()[0] if not x.isdigit() else "NoPrefix")# we used lambda here cuz its like a small method that's gonna take each element in the array and extracts prefix
df["Deck"]=df["Cabin"].str[0]# Ive decided to drop cabin and extract the first letter indicating the deck to kinda prevent overfitting later when i encode them
df=df.drop(columns=["Name","Ticket","Cabin","PassengerId"])
df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,Ticketcount,Ticketprefix,Deck
0,0,3,male,22.0,1,0,7.2500,S,Mr,1,A/5,B
1,1,1,female,38.0,1,0,71.2833,C,Mrs,1,PC,C
2,1,3,female,26.0,0,0,7.9250,S,Miss,1,STON/O2.,B
3,1,1,female,35.0,1,0,53.1000,S,Mrs,2,NoPrefix,C
4,0,3,male,35.0,0,0,8.0500,S,Mr,1,NoPrefix,B
...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Rev,1,NoPrefix,B
887,1,1,female,19.0,0,0,30.0000,S,Miss,1,NoPrefix,B
888,0,3,female,28.0,1,2,23.4500,S,Miss,2,W./C.,B
889,1,1,male,26.0,0,0,30.0000,C,Mr,1,NoPrefix,C


# Questions:


*   What happens to the model if you keep the names and ticket as they are even though you extracted imp features from it?
*   Is taking out the ticket prefix really helpfull?and did these engineered features contribute nothing/



In [ ]:
# handling outliers
import numpy as np
from scipy.stats import zscore
import matplotlib.pyplot as plt
z_score=zscore(df["Fare"])
outliers=np.abs(z_score)>3
dfo=df[outliers]
print(dfo) # it turns out we have only 20 rows out 891 rows that are outliers their fare was unusual i chose to ignore them

     Survived  Pclass     Sex   Age  SibSp  Parch      Fare Embarked Title  \
27          0       1    male  19.0      3      2  263.0000        S    Mr   
88          1       1  female  23.0      3      2  263.0000        S  Miss   
118         0       1    male  24.0      0      1  247.5208        C    Mr   
258         1       1  female  35.0      0      0  512.3292        C  Miss   
299         1       1  female  50.0      0      1  247.5208        C   Mrs   
311         1       1  female  18.0      2      2  262.3750        C  Miss   
341         1       1  female  24.0      3      2  263.0000        S  Miss   
377         0       1    male  27.0      0      2  211.5000        C    Mr   
380         1       1  female  42.0      0      0  227.5250        C  Miss   
438         0       1    male  64.0      1      4  263.0000        S    Mr   
527         0       1    male  28.0      0      0  221.7792        S    Mr   
557         0       1    male  28.0      0      0  227.5250     

In [ ]:
print(df["Title"].unique())


['Mr' 'Mrs' 'Miss' 'Master' 'Don' 'Rev' 'Dr' 'Mme' 'Ms' 'Major' 'Lady'
 'Sir' 'Mlle' 'Col' 'Capt' 'the Countess' 'Jonkheer']


In [ ]:
#split the target "survived" from the rest of the dataset

x=df.drop("Survived",axis=1)
y=df["Survived"]
x

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,Ticketcount,Ticketprefix,Deck
0,3,male,22.0,1,0,7.2500,S,Mr,1,A/5,B
1,1,female,38.0,1,0,71.2833,C,Mrs,1,PC,C
2,3,female,26.0,0,0,7.9250,S,Miss,1,STON/O2.,B
3,1,female,35.0,1,0,53.1000,S,Mrs,2,NoPrefix,C
4,3,male,35.0,0,0,8.0500,S,Mr,1,NoPrefix,B
...,...,...,...,...,...,...,...,...,...,...,...
886,2,male,27.0,0,0,13.0000,S,Rev,1,NoPrefix,B
887,1,female,19.0,0,0,30.0000,S,Miss,1,NoPrefix,B
888,3,female,28.0,1,2,23.4500,S,Miss,2,W./C.,B
889,1,male,26.0,0,0,30.0000,C,Mr,1,NoPrefix,C


In [ ]:


rare_titles=x['Title'].value_counts()[x['Title'].value_counts()<10].index
x["Title"]=x['Title'].apply(lambda x:"rare" if x in rare_titles else x)

In [ ]:
ticket_count=x['Ticketprefix'].value_counts()
rare_ticket=ticket_count[ticket_count<10].index
x["Ticketprefix"]=x['Ticketprefix'].replace(rare_ticket,"rare")
rare_ticket=x['Ticketprefix'].value_counts()[x['Ticketprefix'].value_counts()<10].index
print(x["Ticketprefix"].unique())

['A/5' 'PC' 'rare' 'NoPrefix' 'C.A.' 'STON/O']


In [ ]:
#Encoding features
from sklearn.preprocessing import LabelEncoder
encoders={}
categorical_cols = ['Sex', 'Embarked', 'Title', 'Ticketprefix', 'Deck']
for col in categorical_cols:
    le = LabelEncoder()
    x[col] = le.fit_transform(x[col].astype(str))
    encoders[col]=le#save encoder to use later for test

x.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,Ticketcount,Ticketprefix,Deck
0,3,1,22.0,1,0,7.2500,2,2,1,0,1
1,1,0,38.0,1,0,71.2833,0,3,1,3,2
2,3,0,26.0,0,0,7.9250,2,1,1,5,1
3,1,0,35.0,1,0,53.1000,2,3,2,2,2
4,3,1,35.0,0,0,8.0500,2,2,1,2,1


In [ ]:
print(le.classes_)

['A' 'B' 'C' 'D' 'E' 'F' 'G' 'T']


In [ ]:
#Standarize so that the numbers are in a similar range
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
x_new=sc.fit_transform(x)
x_new

array([[ 0.82737724,  0.73769513, -0.56573646, ..., -0.57916179,
        -2.21981017, -0.34800107],
       [-1.56610693, -1.35557354,  0.66386103, ..., -0.57916179,
         0.49950815,  0.74379101],
       [ 0.82737724, -1.35557354, -0.25833709, ..., -0.57916179,
         2.31238703, -0.34800107],
       ...,
       [ 0.82737724, -1.35557354, -0.1046374 , ...,  0.15592818,
         2.31238703, -0.34800107],
       [-1.56610693,  0.73769513, -0.25833709, ..., -0.57916179,
        -0.40693129,  0.74379101],
       [ 0.82737724,  0.73769513,  0.20276197, ..., -0.57916179,
        -0.40693129, -0.34800107]])

# Model and training


*   Since the target is 0 or 1 we will use logistic regression




In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

x_train,x_val,y_train,y_val=train_test_split(x_new,y,test_size=0.2,random_state=42)
model=LogisticRegression(max_iter=1000)
model.fit(x_train,y_train)
pred=model.predict_proba(x_val)
loss=log_loss(y_val,pred)

print("loss: ", loss)


loss:  0.41273320281532105


Always use predict_proba with log loss cuz i used predict normally and i got a fatal 6.64 log loss which is extremly bad.


*   Look into feature scaling, and regularization to tey and make it better
*  What is a baseline and how to implement it? +how can i make my model better?



*   The loss is 0.41 which means thjat the model is performing ok but it can be better
*   I will use GridSearch to seea difference





In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

x_train,x_val,y_train,y_val=train_test_split(x_new,y,test_size=0.2,random_state=42)
model=LogisticRegression(max_iter=1000)
param_grid = [
    {'penalty': [ 'l2']}]
grid=GridSearchCV(estimator=model,param_grid=param_grid,verbose=1)
grid.fit(x_train,y_train)
pred=grid.predict_proba(x_val)
pred1=grid.predict(x_val)
loss=log_loss(y_val,pred)
acc=accuracy_score(y_val,pred1)
print("loss: ", loss)
print("accuracy: ",acc)


Fitting 5 folds for each of 1 candidates, totalling 5 fits
loss:  0.41273320281532105
accuracy:  0.8100558659217877


I still ended up with the same loss why? and did gridsearch really fix or add any value or was it just another way to implement logistic regression with just a tiny costumization? Also what are real ways that I can actually better my model with?

It is common for GridSearchCV to yield the same (or worse) results on a test set, particularly if the default hyperparameters are already near-optimal for the data, or if the model itself is not suitable for the underlying data structure. Grid search is not a "miracle" tool, but rather a methodical, systematic approach to exploring hyperparameter space—specifically regularization (C and penalty) for logistic regression.

2. Did GridSearch Add Value?
Yes, it added value by guaranteeing you have the best parameters from the range you defined and validating that the default was optimal. It acts as a sanity check. If the GridSearch returns the same hyperparameters as your baseline, you know that tuning C is not the answer.

3. Real Ways to Improve Your Model
If GridSearchCV did not help, you must look outside of hyperparameter tuning.

Feature Engineering (The Most Impactful)

Feature Scaling: Logistic regression requires scaling (e.g., StandardScaler) for regularization to work effectively.

Handle Class Imbalance
Data Quality Analysis
Try Other Algorithms: If linear separation isn't working, try:
Random Forests/Gradient Boosting (XGBoost/LightGBM): These handle non-linear relationships better.
Support Vector Machines (SVM): Better if you need a flexible decision boundary in high-dimensional spaces.
Increase Training Data: Sometimes, you just need more data for the model to learn the underlying pattern.

# Testing The Model

In [ ]:
import pandas as pd
test=pd.read_csv('test.csv')
test# the diff bet it and the train dataset is that it doesnt have the survived column

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [ ]:
dt=test.drop('PassengerId',axis=1)
dt

,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...
413,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [ ]:
#replacing the null values of the column Age

dt["Age"]=dt["Age"].fillna(value=median)#replaces the null values with the median
dt.isnull().any()#shows the columns that have null to check if age is still one of them ,which it is not
dt["Age"].isnull()

,Age
0,False
1,False
2,False
3,False
4,False
...,...
413,False
414,False
415,False
416,False


In [ ]:
#repalcing the null values of the cabin column

dt["Cabin"]=dt["Cabin"].fillna(value=mode_c)

dt["Embarked"]=dt["Embarked"].fillna(value=mode_e)
dt.drop_duplicates()
dt["Title"]=dt["Name"].str.split(",").str[1].str.split('.').str[0].str.strip()## Extract passenger titles from names
dt["Ticketcount"]=dt.groupby("Ticket")["Ticket"].transform('count')# groups passengesrrs that had the same tickets /basically they were traveling together or not
dt["Ticketprefix"]=dt["Ticket"].apply(lambda x: x.split()[0] if not x.isdigit() else "NoPrefix")# we used lambda here cuz its like a small method that's gonna take each element in the array and extracts prefix
dt["Deck"]=dt["Cabin"].str[0]# Ive decided to drop cabin and extract the first letter indicating the deck to kinda prevent overfitting later when i encode them
dt=dt.drop(columns=["Name","Ticket","Cabin"])


dt



,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,Ticketcount,Ticketprefix,Deck
0,3,male,34.5,0,0,7.8292,Q,Mr,1,NoPrefix,B
1,3,female,47.0,1,0,7.0000,S,Mrs,1,NoPrefix,B
2,2,male,62.0,0,0,9.6875,Q,Mr,1,NoPrefix,B
3,3,male,27.0,0,0,8.6625,S,Mr,1,NoPrefix,B
4,3,female,22.0,1,1,12.2875,S,Mrs,1,NoPrefix,B
...,...,...,...,...,...,...,...,...,...,...,...
413,3,male,28.0,0,0,8.0500,S,Mr,1,A.5.,B
414,1,female,39.0,0,0,108.9000,C,Dona,1,PC,C
415,3,male,38.5,0,0,7.2500,S,Mr,1,SOTON/O.Q.,B
416,3,male,28.0,0,0,8.0500,S,Mr,1,NoPrefix,B


In [ ]:
dt.isnull().any()

,0
Pclass,False
Sex,False
Age,False
SibSp,False
Parch,False
Fare,True
Embarked,False
Title,False
Ticketcount,False
Ticketprefix,False


In [ ]:
medianF=dt["Fare"].median()
dt["Fare"]=dt["Fare"].fillna(value=medianF)
dt.isnull().any()

,0
Pclass,False
Sex,False
Age,False
SibSp,False
Parch,False
Fare,False
Embarked,False
Title,False
Ticketcount,False
Ticketprefix,False


In [ ]:
print(encoders['Sex'].classes_)

['female' 'male']


In [ ]:
print(df["Title"].unique())
print(dt["Title"].unique())

['Mr' 'Mrs' 'Miss' 'Master' 'Don' 'Rev' 'Dr' 'Mme' 'Ms' 'Major' 'Lady'
 'Sir' 'Mlle' 'Col' 'Capt' 'the Countess' 'Jonkheer']
['Mr' 'Mrs' 'Miss' 'Master' 'Ms' 'Col' 'Rev' 'Dr' 'Dona']


In [ ]:

dt["Title"]=dt['Title'].apply(lambda val:val if val in encoders['Title'].classes_ else "rare")
print(dt["Title"].unique())

['rare']


In [ ]:
dt['Ticketprefix'].unique()

array(['NoPrefix', 'A/4', 'W.E.P.', 'SC/PARIS', 'STON/O2.', 'PC', 'C',
       'A/5.', 'SC/AH', 'C.A.', 'W./C.', 'SOTON/O.Q.', 'STON/O', 'SC/A.3',
       'F.C.C.', 'F.C.', 'A./5.', 'PP', 'STON/OQ.', 'SOTON/OQ', 'CA',
       'SC/A4', 'S.O./P.P.', 'CA.', 'S.O.C.', 'SOTON/O2', 'AQ/4', 'A.',
       'SC', 'A/5', 'SC/Paris', 'LP', 'AQ/3.', 'S.C./PARIS', 'A.5.'],
      dtype=object)

In [ ]:

dt["Ticketprefix"]=dt['Ticketprefix'].apply(lambda val:val if val in encoders['Ticketprefix'].classes_ else "rare")
print(dt["Ticketprefix"].unique())

['rare']


In [ ]:
print(x['Deck'].unique())
print(dt['Deck'].unique())

[1 2 4 6 3 0 5 7]
['B' 'E' 'A' 'C' 'D' 'F' 'G']


In [ ]:
#categorical_cols = ['Sex', 'Embarked', 'Title', 'Ticketprefix', 'Deck']
for col in categorical_cols:
    dt[col] = encoders[col].transform(dt[col].astype(str))

#never use fit_transform on data since it means that you are gonna learn from it which shouldnt be the case
dt=dt[x.columns]
dt=sc.transform(dt)
dt

array([[ 0.82737724,  0.73769513,  0.39488658, ..., -0.57916179,
         2.31238703, -0.34800107],
       [ 0.82737724, -1.35557354,  1.35550962, ..., -0.57916179,
         2.31238703, -0.34800107],
       [-0.36936484,  0.73769513,  2.50825727, ..., -0.57916179,
         2.31238703, -0.34800107],
       ...,
       [ 0.82737724,  0.73769513,  0.70228595, ..., -0.57916179,
         2.31238703, -0.34800107],
       [ 0.82737724,  0.73769513, -0.1046374 , ..., -0.57916179,
         2.31238703, -0.34800107],
       [ 0.82737724,  0.73769513, -0.1046374 , ..., -0.57916179,
         2.31238703, -0.34800107]])

# Testing

In [ ]:
test_predictions=grid.predict(dt)
test_predictions

array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0,
       1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1,
       1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
       0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
       1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [ ]:
print(len(test))
print(len(test_predictions))
print(len(dt))
print(len(test['PassengerId']))

418
418
418
418


In [ ]:
submission=pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_predictions
})
submission.to_csv("submission.csv",index=False)

In [ ]:
submission

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


## Final Results
- Validation Accuracy: 0.81
- Validation Log Loss: 0.41
- Kaggle Public Score: 0.77033


## Conclusion

This project explored logistic regression for Titanic survival prediction.
Feature engineering significantly improved performance, especially Titles and Deck extraction.
The project also highlighted the importance of consistent preprocessing between train and test data.